In [ ]:
# Load libraries and set up environment
import os 
import sys
import importlib
import datetime
import numpy as np
import pandas as pd
import anndata as ad    
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import gffutils
import anndata as ad
import scanpy as sc 
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from mord import OrdinalRidge  
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch 

# Configure plotting styles
sns.set_theme()
sc.set_figure_params(figsize=(7, 7), frameon=True, dpi=80, facecolor='white')

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

# Define module paths
src_path = "/gpfs/commons/home/kisaev/Leaflet-private/src/"

# Add to sys.path if not already present
if src_path not in sys.path:
    sys.path.append(src_path)

# Import custom modules
import BetaDirichletFactor.LeafletFA as LeafletFA
import BetaDirichletFactor.differential_splicing as ds
import BetaDirichletFactor.utils as utils
import visualization.IsovizPy as ja
import evaluations.cost_correlation_assign as cca

# Reload custom modules
importlib.reload(LeafletFA)
importlib.reload(ds)
importlib.reload(utils)

# Import functions from utilities.py 
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-analysis')
from LeafletFA_utilities import *

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import glob
import os
import lzma
import pickle

### Identify best model fit

In [ ]:
model_outputs = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/leafletFAmodel/2025-02-21/"

# Check the specific pattern
run_summary_files = glob.glob(os.path.join(model_outputs, "*", "*run_summary.csv"))

# Read in the run summary files
run_summary = pd.concat([pd.read_csv(f) for f in run_summary_files])
run_summary.sort_values(by="best_elbo", ascending=True)

In [ ]:
best_run = run_summary.sort_values(by="best_elbo", ascending=True).iloc[0]
print(f"The best run is {best_run['param_id']} with an ELBO of {best_run['best_elbo']}")

### Load trained model latent variables

In [ ]:
# Define paths
# run outputs is model_outputs followed by dir "run_param_id[0]"

run_outputs = os.path.join(model_outputs, f"run_{best_run['param_id']}")
model_file = os.path.join(run_outputs, "leafletfa_model.pkl.xz")

# Load the model
def load_model(model_file):
    with lzma.open(model_file, "rb") as f:
        model = pickle.load(f)
    return model

# Load and store the model
leaflet_model = load_model(model_file)

# Debug
print(f"Model loaded successfully from {model_file}")

In [ ]:
importlib.reload(ds)
factor_idx=0
ds.analyze_differential_splicing(leaflet_model)

### Load Anndata

In [ ]:
### Load the original anndata object 
ATSE_anndata_file = "/gpfs/commons/groups/knowles_lab/Karin/TMS_MODELING/DATA_FILES/ALL_CELLS/022025/TMS_Anndata_ATSE_counts_with_waypoints_20250209_165655.h5ad"

print(f"Loading Anndata file: {ATSE_anndata_file}")
adata = ad.read_h5ad(ATSE_anndata_file)
print(f"Anndata file loaded successfully.")

### Add latent variables to anndata 

In [ ]:
### Load the latent variables
PHI = leaflet_model.assign_post # (C by K) 
PSI = leaflet_model.psi_learned # (K by J)
PI = leaflet_model.pi # (K, )
K = leaflet_model.K

In [ ]:
# Make a quick barplot of PI 
PI_df = pd.DataFrame(PI, columns=["PI"])
PI_df["Factor"] = PI_df.index
PI_df = PI_df.sort_values(by="PI", ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x="Factor", y="PI", data=PI_df, order=PI_df["Factor"])
plt.title("Factor K Probabilities (Sorted)")
plt.xlabel("Factor K")
plt.ylabel("Global Assignment Probability")
plt.axhline(0.01, color="red", linestyle="--")
plt.xticks(rotation=90)  # Rotate x-axis labels if many factors

In [ ]:
# Add PHI to adata 
adata.obsm["X_PHI"] = PHI
sc.pp.neighbors(adata, use_rep="X_PHI")
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=["tissue", "age"], wspace=0.7, size=8)

### LeafletFA vs aging regression 

In [ ]:
adata.obs["age_group"] = adata.obs["age"].str.replace("m", "").astype(int)

# Add old/young label
create_age_category(adata)

In [ ]:
# Define the target variable
y = adata.obs["age_group"]
X = adata.obsm["X_PHI"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Compute correlation matrix
corr_matrix = np.corrcoef(X_train.T)  # K × K

# Generate factor labels
factor_labels = [f"factor_{i}" for i in range(X_train.shape[1])]

sns.clustermap(corr_matrix, cmap="coolwarm", center=0, xticklabels=factor_labels, yticklabels=factor_labels, figsize=(8, 8))

# Add title
plt.suptitle("Clustered Correlation Matrix of \nLatent Factors (X_PHI)", y=1.05)
plt.show()


In [ ]:
# Train the ordinal regression model
ordinal_model = OrdinalRidge(alpha=10.0)
ordinal_model.fit(X_train, y_train)
ordinal_coeffs = ordinal_model.coef_

# Bootstrapped Confidence Intervals
n_bootstraps = 100
bootstrap_coefs = np.zeros((n_bootstraps, X_train.shape[1]))

for i in tqdm(range(n_bootstraps)):
    X_resampled, y_resampled = resample(X_train, y_train, random_state=i)
    model = OrdinalRidge(alpha=10.0)
    model.fit(X_resampled, y_resampled)
    bootstrap_coefs[i, :] = model.coef_

# Compute 95% confidence intervals
ci_lower = np.percentile(bootstrap_coefs, 2.5, axis=0)
ci_upper = np.percentile(bootstrap_coefs, 97.5, axis=0)

# Factor names
factor_labels = [f"factor_{i}" for i in range(len(ordinal_coeffs))]

# Sorting factors by absolute importance (optional)
sorted_indices = np.argsort(np.abs(ordinal_coeffs))[::-1]
ordinal_coeffs = ordinal_coeffs[sorted_indices]
ci_lower = ci_lower[sorted_indices]
ci_upper = ci_upper[sorted_indices]
factor_labels = [factor_labels[i] for i in sorted_indices]

# Plot with vertical layout
plt.figure(figsize=(7, 10))
plt.barh(factor_labels, ordinal_coeffs, color="blue", alpha=0.7)
plt.errorbar(ordinal_coeffs, factor_labels, 
             xerr=[ordinal_coeffs - ci_lower, ci_upper - ordinal_coeffs], 
             fmt='o', color='black', capsize=2)

plt.axvline(0, color="black", linestyle="dashed")  # Dashed line at 0
plt.xlabel("Coefficient")
plt.ylabel("Latent Factor")
plt.title("Ordinal Ridge Regression Coefficients with 95% CI (Predicting Age)")
plt.gca().invert_yaxis()  # Invert so most important factor is at the top
plt.show()

In [ ]:
# Extract factor matrix
X_PHI = adata.obsm["X_PHI"]

# Convert X_PHI to a DataFrame and add age labels
factor_df = pd.DataFrame(X_PHI, columns=factor_labels, index=adata.obs.index)
factor_df["age_category"] = adata.obs["age_category"].values  # Add age labels

# Compute median factor activity per age category
age_groups = factor_df.groupby('age_category')[factor_labels].median()

# Ensure 'old' and 'young' exist before computing deltas
if 'old' in age_groups.index and 'young' in age_groups.index:
    deltas = pd.DataFrame({
        'Factor': factor_labels,
        'Delta_OY': age_groups.loc['old'] - age_groups.loc['young']
    })
else:
    raise ValueError("Missing 'old' or 'young' category in age groups.")

# Create DataFrame for ordinal regression coefficients
ordinal_coeffs_df = pd.DataFrame({
    "Factor": factor_labels,
    "Coefficient": ordinal_coeffs,
    "CI_Lower": ci_lower,
    "CI_Upper": ci_upper
})

# Merge deltas with coefficients
coefficients_with_deltas = deltas.merge(ordinal_coeffs_df, on="Factor")


In [ ]:
plot_factors_with_delta(coefficients_with_deltas=coefficients_with_deltas, highlight_factors=["factor_0", "factor_10", "factor_18"])

In [ ]:
plot_top_factor_distributions(adata, [0, 1, 10, 18], age_column="age")

In [ ]:
# confirm that all rows sum to 1 in PHI 
assert np.allclose(np.sum(PHI, axis=1), 1.0)

### Train multiclass model to predict cell type 

In [ ]:
cell_type_accuracy, model, encoder, coef_df = logistic_regression_feature_prediction_simple(adata, "cell_type_grouped")

In [ ]:
plot_clustermap(coef_df, highlighted_factors=["factor_0", "factor_10"], cmap="seismic", figsize=(10, 10), center=0)

In [ ]:
# let's look at all factors 
factors_to_compare = list(range(K))
result_df = compare_factors_by_age(adata, factors_to_compare, cell_type_col="cell_type_grouped")

In [ ]:
# Step 1: Create a pivot table for delta_median and avg_factor_activity
delta_median_pivot = result_df.pivot(index='cell_type', columns='factor', values='delta_median')
avg_activity_pivot = result_df.pivot(index='cell_type', columns='factor', values='avg_factor_activity')
median_activity_pivot = result_df.pivot(index='cell_type', columns='factor', values='median_factor_activity')

# Step 3: Define a colormap that includes black for masked cells
cmap = sns.color_palette("seismic", as_cmap=True)
cmap.set_bad(color='grey')  # Set masked values to black

# Step 3: Cluster the delta_median_pivot to get row and column order
g = sns.clustermap(delta_median_pivot, cmap="seismic", cbar=True, 
                   annot=False, center=0, vmin=-0.05, vmax=0.05, figsize=(4, 4),
                   row_cluster=True, col_cluster=True)

In [ ]:
# Extract the cluster order from the clustermap
cell_type_order = g.dendrogram_row.reordered_ind
factor_order = g.dendrogram_col.reordered_ind

# Reindex avg_activity_pivot to match the clustered order of delta_median_pivot
avg_activity_pivot = avg_activity_pivot.iloc[cell_type_order, factor_order]
delta_median_pivot = delta_median_pivot.iloc[cell_type_order, factor_order]  # Reorder this as well

# Step 4: Create a figure for two side-by-side heatmaps
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Step 5: Plot the delta_median_pivot heatmap with the clustering-based order
sns.heatmap(delta_median_pivot, cmap="seismic", ax=ax1, cbar=True, 
            annot=False, center=0, vmin=-0.05, vmax=0.05, xticklabels=True, yticklabels=True)

# Step 6: Plot the avg_activity_pivot heatmap with the same order as delta_median_pivot
sns.heatmap(avg_activity_pivot, cmap="coolwarm", ax=ax2, cbar=True, 
            annot=False, center=0, vmin=avg_activity_pivot.min().min(), vmax=avg_activity_pivot.max().max(), 
            xticklabels=True, yticklabels=False)  # Show x-axis ticks but hide y-axis ticks

# Step 7: Set titles for the plots
ax1.set_title('Delta Median Factor Activity (Old - Young)', fontsize=10)
ax2.set_title('Average Factor Activity in Cell Type', fontsize=10)

# Step 8: Set x-axis and y-axis labels for the left plot (since it's clustered) 
ax1.set_xticklabels(delta_median_pivot.columns, rotation=90, fontsize=10)
ax1.set_yticklabels(delta_median_pivot.index, rotation=0, fontsize=10)
ax1.set_ylabel("Cell Type")
ax1.set_xlabel("Factor")

# Set x-axis labels for the right plot (factors should be shown)
ax2.set_xticklabels(avg_activity_pivot.columns, rotation=90, fontsize=10)
ax2.set_ylabel("Cell Type")
ax2.set_xlabel("Factor")

# Adjust layout
plt.tight_layout()

# Step 5: Save the figure as a PDF with filename including current date
current_date = datetime.now().strftime("%Y-%m-%d")
filename = f"supp_heatmaps_delta_avg_factors_{current_date}.pdf"
plt.savefig(filename, format='pdf')

plt.show()

In [ ]:
# Automatically filter and plot factor_20
plot_factor_medians(result_df, "factor_15")  # Set a path to save, or None to just display

In [ ]:
plot_factor_distribution(adata, "NEURON", 16)

### Load gene expression 

In [ ]:
ge_adata = "/gpfs/commons/groups/knowles_lab/Karin/TMS_MODELING/DATA_FILES/ALL_CELLS/022025/TMS_Anndata_GeneExpression_20250209_165655.h5ad"

# Load the gene expression anndata object
ge_adata = ad.read_h5ad(ge_adata)

In [ ]:
ge_adata

In [ ]:
accuracies, models, label_encoder, coefficients_dfs = logistic_regression_feature_prediction(
    splice_adata=adata,
    gene_expression_adata=ge_adata,
    feature="age",
    K=50,
    n_pcs=50,
    covariate_column="tissue",
    test_size=0.2
)

In [ ]:
print(accuracies)

In [ ]:
coefficients_dfs

### Differential Splicing Analysis 

In [ ]:
if K > 2:
    results = ds.analyze_all_factors(leaflet_model, fdr_threshold=0.05, min_effect_size=0.1)

    # Let's collect all the dataframes from results
    # go through results.keys() and collect second value in tuple and add column indicating factor K
    results_df = pd.DataFrame()
    for key in results.keys():
        factor_res = results[key][1]
        factor_res["factor_K"] = key
        results_df = pd.concat([results_df, factor_res])

    # for every unique junctions i want to summarize number of factors that it was significant in
    sig_only = results_df[results_df["significant"] == True]
    sig_only = sig_only.groupby("junction_idx").agg({"significant": "count"}).reset_index()
    sig_only = sig_only.rename(columns={"significant": "num_factors_significant"})
    # merge with results_df
    results_df = results_df.merge(sig_only, on="junction_idx", how="left")
    results_df["abs_effect_size"] = np.abs(results_df["effect_size"])
